The contents of this directory benchmark various model approaches. Results are saved as
1. Actual models
2. Slurm logs
3. HTML Dask performance reports

# Imports

In [1]:
# Imports
import subprocess
from datetime import datetime
import pickle
import os

import numpy as np
import pandas as pd

import statsmodels
from statsmodels.discrete.count_model import ZeroInflatedNegativeBinomialResultsWrapper as smzinb

# Submitting model runs

In [4]:
#functions for model submission...

data_root="/gpfs/gibbs/pi/reilly/tabula_data"

def bench(model_code, t):
    """
    Executes a particular model design & collects statistics. 
    t is time in hours
    """
    now=datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    
    logdir=f"{data_root}/speed_test/logs/{model_code}_{now}"

    #make a directory to put all the log files : this will be 
    os.makedirs(logdir, exist_ok=True)

    command=f"""
    module load miniconda
    conda activate env_tensorzinb_cuda
    code_location=$(pwd)
    cd {logdir}
    python ${{code_location}}/cluster.py {model_code}
    """


    slurm_cmd = [
        "sbatch",
        "--partition=ycga",
        f"--time={t}:00:00",
        f"--output={logdir}/master_{model_code}_{now}.out",
        "-c 1",
        f"-J {model_code}_master",
        "--wrap", command
    ]
    result = subprocess.run(
        slurm_cmd, 
        capture_output=True, 
        text=True
    )
    
    print(result.stdout.strip() if result.returncode == 0 else result.stderr.strip())



In [5]:
#bench("c0011000",12)

Submitted batch job 50981475


In [23]:
#bench("c9011090",1)

Submitted batch job 50976899


In [8]:
#bench("c9100090",1)

Submitted batch job 50973661


In [12]:
#done
#bench("c9100010",1)
#bench("c0200010",12)
#bench("c9100090",1)
#bench("c0100010",12)
#bench("c9200090",1)
#bench("c9200010",1)
#bench("c0100000",12)
#bench("c0200000",12)

Submitted batch job 50854194


In [6]:
#currently running
bench("c0100000",12)
bench("c0200000",12)


Submitted batch job 50958038
Submitted batch job 50958039


# Summarizing success / failure
& everything in-between

In [14]:
def extract_parameters():
    """
    Takes a model & extracts triplets (pi, sigma-squared, mu)
    (break by type...)
    unimplemented
    """
    pass

def load_model(model_file:str):
    """
    What is says on the tin. Points to relevant archive path to avoid retyping.
    """
    with open(f"{data_root}/speed_test/models/arch/{model_file}","rb") as f:
        return pickle.load(f)

def eval_single_statsmodels(model):
    """
    Takes a single statsmodels model & returns some useful QC metrics
    """
    cov=-1
    finite_cov=-1
    try:
        cov = model.cov_params()
        finite_cov = np.all(np.isfinite(cov.values))
    except (ValueError, np.linalg.LinAlgError) as e:
        finite_cov = False
        cov = None

    converged = model.mle_retvals.get("converged", False)
    finite_se = np.all(np.isfinite(model.bse)) if hasattr(model, "bse") else False

    return {
        "converged": converged,
        "covariance_matrix_available": cov is not None,
        "covariance_matrix_finite": finite_cov,
        "standard_errors_finite": finite_se
        #"cov_type": getattr(model, "cov_type", None),
    }

def infer_converged_from_weights(model, tolerance=1e-3):
    all_weights = []
    for w in model.get('weights', {}).values():
        if isinstance(w, np.ndarray):
            all_weights.append(w.flatten())
    
    if not all_weights:
        return False  # no weights found, assume not converged

    all_weights = np.concatenate(all_weights)

    std_dev = np.std(all_weights)

    return std_dev > tolerance

def eval_single_tensorzinb(model):
    """
    Takes a single tensorzinb model & returns some useful QC metrics. 
    """

    return {
        "converged": infer_converged_from_weights(model),
        "covariance_matrix_available": None,
        "covariance_matrix_finite": None,
        "standard_errors_finite": None
        #"cov_type": getattr(model, "cov_type", None),
    }
    pass

def eval_model(model):
    """
    Produces QC metrics on a single model, regardless of type
    """
    #type checking is done here
    
    if isinstance(model,smzinb):
        #if its a statsmodels object, it must be a unified statsmodels model
        return pd.DataFrame.from_dict(eval_single_statsmodels(model), orient='index', columns=['only'])
        
    elif isinstance(model,dict):
        #if it's a dict... it could be a broken statsmodels
        # or unified or broken 
        #
        all_zinb = [
            isinstance(model[key], smzinb)
            for key in model
        ]
        if all(all_zinb):
            #must be broken statsmodels
            return pd.DataFrame({
                key:eval_single_statsmodels(model[key])
                for key in model
            })

        else:
            #must be either unified or broken tensorzinb.
            #disambiguate by checking for a sentinel value
            if "multi_sent" in model.keys():
                # broken 
                print("[!] Type not implemented yet. Aborting.")
            else:
                #single tensorzinb model. 
                return eval_single_tensorzinb(model)
    else:
        print("[!] Type not implemented yet. Aborting.")



def eval_models(modelpaths):
    """
    Loads and evaluates models, taking a list of filenames
    and returning a dictionary of model IDs pointing at
    evaluations
    """
    evals={}
    for path in modelpaths:
        name=path.split("_")[0]
        try:
            model=load_model(path)
            evals[name]=eval_model(model)
            del model
        except EOFError:
            print(f"EOF error on {name}")
    return evals

def compare_models(evals):
    """
    Takes output of eval_models and compresses
    into a summary dataframe. 
    """
    summaries=[]
    for name in evals:
        summary=evals[name].sum(axis=1)
        summary["total"]=len(evals[name].columns)
        summary["model"]=name
        summaries.append(summary)
    summaries=pd.DataFrame(summaries)
    summaries.set_index(summaries["model"],inplace=True)
    summaries.drop("model",axis=1,inplace=True)
    return summaries

def eval_all():
    file_names=[f for f in os.listdir(f"{data_root}/speed_test/models/arch/")]
    evals=eval_models(file_names)
    return evals

def summarize_all():
    return compare_models(eval_all())


In [15]:
load_model("c9010090_2025-04-28_15-33-15.pkl")

{'llf_total': -22225.59798024269,
 'llfs': array([-22225.59798024]),
 'aic_total': 44471.19596048538,
 'aics': array([44471.19596049]),
 'df_model_total': 10,
 'df': 10,
 'weights': {'x_mu': array([[ 4.6605802e+00],
         [ 4.0144329e+00],
         [ 3.6819699e-01],
         [ 4.2448468e+00],
         [ 2.4059439e+00],
         [-2.6354466e-03]], dtype=float32),
  'x_pi': array([[ 1.3365092],
         [-1.3360662],
         [ 0.8901649]], dtype=float32),
  'theta': array([[0.5682316]], dtype=float32)},
 'cpu_time': 1.773240089416504,
 'num_sample': 14312,
 'epochs': 543}

In [16]:
eval_model(load_model("c9010090_2025-04-28_15-33-15.pkl"))

{'converged': True,
 'covariance_matrix_available': None,
 'covariance_matrix_finite': None,
 'standard_errors_finite': None}

In [6]:
summary=summarize_all()

EOF error on c0100000
EOF error on c0200000


In [7]:
summary

,converged,covariance_matrix_available,covariance_matrix_finite,standard_errors_finite,total
model,,,,,
c0100010,10,0,0,0,10
c0200010,10,6,6,6,10
c9100010,2,2,2,2,2
c9100090,1,1,1,1,1
c9200010,1,1,1,1,2
c9200090,1,1,1,1,1


Adding metadata so we don't have to memorize all the IDs.

In [8]:
modelspecs=pd.read_csv(f"{data_root}/speed_test/modelspecs.tsv",sep="\t")
modelspecs.set_index(modelspecs["code"],inplace=True)
modelspecs.drop("code",axis=1,inplace=True)

In [24]:
summary=summary.join(modelspecs,how="inner")
summary

,converged,covariance_matrix_available,covariance_matrix_finite,standard_errors_finite,total,dataset,sm_optimizer,lib,hardware,dna,main_equ_type,z_equ_type,main_equ,z_equ,broken_by
c0100010,10,0,0,0,10,Shendure (0),1_bfgs,statsmodels (0),CPU (0),No dna (0),by cell-type (1),replicate (0),umis_mpra_bc ~ C(cre_id)-1,C(rep_id),cell_type
c0200010,10,6,6,6,10,Shendure (0),2_cg,statsmodels (0),CPU (0),No dna (0),by cell-type (1),replicate (0),umis_mpra_bc ~ C(cre_id)-1,C(rep_id),cell_type
c9100010,2,2,2,2,2,Fake (9),1_bfgs,statsmodels (0),CPU (0),No dna (0),by cell-type (1),replicate (0),umi_count ~ C(cre_id)-1,C(rep_id),cell_type
c9100090,1,1,1,1,1,Fake (9),1_bfgs,statsmodels (0),CPU (0),No dna (0),simple interaction (9),replicate (0),umi_count ~ C(cre_id)*C(cell_type)-1,C(rep_id),NaN
c9200010,1,1,1,1,2,Fake (9),2_cg,statsmodels (0),CPU (0),No dna (0),by cell-type (1),replicate (0),umi_count ~ C(cre_id)-1,C(rep_id),cell_type
c9200090,1,1,1,1,1,Fake (9),2_cg,statsmodels (0),CPU (0),No dna (0),simple interaction (9),replicate (0),umi_count ~ C(cre_id)*C(cell_type)-1,C(rep_id),NaN


# Summarizing performance

In [18]:
def format_seconds(seconds):
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = seconds % 60
    return f"{hours}h {minutes}m {secs:.2f}s"

def simple_time(model_file):
    """
    Produces a quick time estimate from log files.
    Not as accurate or detailed as the full HTML report.
    """
    directory=f"{data_root}/speed_test/logs/arch/{model_file}"
    master_files = [f for f in os.listdir(directory) if f.startswith("master")]
    if len(master_files) != 1:
        raise ValueError(f"Expected exactly one 'master*' file, found {len(master_files)}.")
    
    filepath = os.path.join(directory, master_files[0])

    # dump file into memory (its small)
    with open(filepath, 'r') as file:
        lines = file.readlines()

    # Look for the line of interest and extract the last field
    for line in lines:
        if line.startswith("[+] Done with all tasks."):
            return format_seconds(float(line.strip().split()[-1]))
    
    raise ValueError("Could not find final timestamp. Did the model finish?")


In [19]:
def simple_time_all():
    """
    returns a two column data-frame
    index is model code, `time` is time in seconds. 
    """
    directory = f"{data_root}/speed_test/logs/arch/"
    runs = [f for f in os.listdir(directory)]
    result = {}
    for run_name in runs:
        try:
            model_code = run_name.split("_")[0]
            result[model_code] = simple_time(run_name)
        except ValueError:
            print(f"Error on {run_name}")
    return result


In [20]:
times=pd.DataFrame(list(simple_time_all().items()),columns=["model","time"])
times.set_index("model",inplace=True)
times

Error on c0100000_2025-04-25_17-40-02
Error on c0200000_2025-04-25_20-30-58


,time
model,
c0100010,1h 27m 21.26s
c0200010,1h 35m 24.64s
c9100010,0h 0m 32.25s
c9100090,0h 0m 10.53s
c9200010,0h 0m 21.70s
c9200090,0h 0m 23.17s


In [25]:
summary.join(times,how="inner")

,converged,covariance_matrix_available,covariance_matrix_finite,standard_errors_finite,total,dataset,sm_optimizer,lib,hardware,dna,main_equ_type,z_equ_type,main_equ,z_equ,broken_by,time
c0100010,10,0,0,0,10,Shendure (0),1_bfgs,statsmodels (0),CPU (0),No dna (0),by cell-type (1),replicate (0),umis_mpra_bc ~ C(cre_id)-1,C(rep_id),cell_type,1h 27m 21.26s
c0200010,10,6,6,6,10,Shendure (0),2_cg,statsmodels (0),CPU (0),No dna (0),by cell-type (1),replicate (0),umis_mpra_bc ~ C(cre_id)-1,C(rep_id),cell_type,1h 35m 24.64s
c9100010,2,2,2,2,2,Fake (9),1_bfgs,statsmodels (0),CPU (0),No dna (0),by cell-type (1),replicate (0),umi_count ~ C(cre_id)-1,C(rep_id),cell_type,0h 0m 32.25s
c9100090,1,1,1,1,1,Fake (9),1_bfgs,statsmodels (0),CPU (0),No dna (0),simple interaction (9),replicate (0),umi_count ~ C(cre_id)*C(cell_type)-1,C(rep_id),NaN,0h 0m 10.53s
c9200010,1,1,1,1,2,Fake (9),2_cg,statsmodels (0),CPU (0),No dna (0),by cell-type (1),replicate (0),umi_count ~ C(cre_id)-1,C(rep_id),cell_type,0h 0m 21.70s
c9200090,1,1,1,1,1,Fake (9),2_cg,statsmodels (0),CPU (0),No dna (0),simple interaction (9),replicate (0),umi_count ~ C(cre_id)*C(cell_type)-1,C(rep_id),NaN,0h 0m 23.17s
